# Exercise 3 — How locked in are we?

⏱️ 10 minutes investigating, then we discuss as a group.

## The situation

A platform architect posts in your team channel:

> Leadership wants a call by Friday on whether every team standardises on the same setup for
> our tables. The Trino users are pushing for Iceberg, and the streaming team keep mentioning
> Hudi. Before I write the recommendation — you've been building on Delta this week. How
> locked in are we, actually? What would it even cost to move?

Before answering, find out what you've actually been relying on. Start by looking at the
folder your table from Exercise 2 lives in.

In [ ]:
from northtrail import get_spark, path, local_data, ls

spark = get_spark("exercise-3")
ORDERS = path("raw", "orders")

# If you didn't finish Exercise 2, this rebuilds the same table so you can carry on.
try:
    spark.sql(f"DESCRIBE HISTORY delta.`{ORDERS}`").count()
except Exception:
    print("Rebuilding the Exercise 2 table...")
    (spark.read.parquet(local_data("lake", "raw", "orders"))
          .repartition(3).write.mode("overwrite").parquet(ORDERS))
    spark.sql(f"CONVERT TO DELTA parquet.`{ORDERS}`")
    spark.sql(f"UPDATE delta.`{ORDERS}` SET amount = round(amount / 100, 2) WHERE amount > 1000")
    spark.sql(f"DELETE FROM delta.`{ORDERS}` WHERE customer_id = 'TEST'")

for entry in ls(spark, ORDERS):
    print(entry)

Two kinds of thing: the Parquet data files, and one extra folder. Look inside that folder.

In [ ]:
for entry in ls(spark, f"{ORDERS}/_delta_log"):
    print(entry)

In [ ]:
# Open the commit written when you corrected the three prices.
#TODO: Replace the ??? with appropriate values for the Delta log JSON file.
import json

for row in spark.read.text(f"{ORDERS}/_delta_log/???.json").collect():
    (kind, body), = json.loads(row.value).items()
    interesting = {k: v for k, v in body.items()
                   if k in ("path", "size", "dataChange", "operation", "operationParameters")}
    print(f"{kind:12} {json.dumps(interesting)[:180]}")

## 🔍 What just happened?

That's it. That's the whole mechanism.

A commit is a small JSON file that names which data files stop counting as part of the table
and which start counting. Your `UPDATE` never edited a Parquet file — it wrote a new one and
recorded the swap. The "table" is the result of replaying these files in order.

Which also explains something you haven't used yet: if the current state is just a replay of
commits 0 to N, then stopping at commit N-1 gives you the table as it was before your update.
Nothing needs to be restored from anywhere. Hold that thought for the next exercise.

So — what have you actually committed to here? A JSON file layout.

## 💡 Concept: open table formats

**Delta Lake, Apache Iceberg and Apache Hudi are specifications, not software you run.** Each
one documents a metadata layout that turns a folder of Parquet files into a table. Your data
is Parquet either way; the format only describes the bookkeeping folder beside it.

They differ in how that bookkeeping is arranged, and those differences follow from what each
was built to do well — cheap file pruning without listing storage, or frequent record-level
updates, or a log simple enough to read by eye. None of them is a service that sits between
you and your data.

Open `reference/comparison.md` alongside the folder listings below.

In [ ]:
# The same table, as Iceberg and as Hudi would lay it out. Illustrative files -- nothing runs them.
from pathlib import Path

REF = next(p for p in [Path("reference"), Path("../reference")] if p.is_dir())

for tree in ["iceberg-example", "hudi-example"]:
    print(f"\n{tree}/")
    for f in sorted((REF / tree).rglob("*")):
        if f.is_file():
            print(f"   {f.relative_to(REF / tree)}")

In [ ]:
# One Iceberg metadata file, and one Hudi timeline entry.
print((REF / "iceberg-example/metadata/v2.metadata.json").read_text()[:900])
print("\n" + "=" * 70 + "\n")
print((REF / "hudi-example/.hoodie/README.txt").read_text())

## Your turn

Three teams, three workloads. Match each to a format and give **one line** of justification.
Go with your gut — you get to argue it properly in the final exercise.

| Workload | Format | Because |
|---|---|---|
| **A.** Change-data-capture from 40 OLTP databases, read by Trino, Flink and Spark. | | |
| **B.** Databricks-only shop. BI dashboards plus a bit of ML. Platform team is two people. | | |
| **C.** Real-time analytics. Thousands of small updates to existing records per minute. | | |

## Debrief

- Answer the architect: if you had to move this table to a different format next quarter,
  what would actually have to happen, and what part of it would hurt?
- "Whatever your platform already supports" is often the right answer in practice. When is
  that reasoning lazy, and when is it correct?
- All three formats keep your data in open Parquet files. So where can a vendor still lock
  you in?